# Ratefluencer Model Training

This notebook generates a synthetic dataset of influencers, computes features (authenticity, engagement, growth), trains a LightGBM regressor to predict a combined Ratefluencer score, and saves a model to `backend/ml/ratefluencer_model.joblib`.

In [ ]:
# 1. Imports
import os
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
print('pandas', pd.__version__)

In [ ]:
# 2. Generate synthetic dataset and preview
from train_ratefluencer import generate_synthetic_influencers
df = generate_synthetic_influencers(300)
df.head()

In [ ]:
# 3. Train a LightGBM model
X = df[['followers','avgLikes','avgComments','engagement_rate','authenticity','growth_score']]
y = df['ratefluencer_score']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val)
params = {'objective':'regression','metric':'rmse','verbosity':-1}
model = lgb.train(params, train_data, valid_sets=[val_data], num_boost_round=100, early_stopping_rounds=10)
joblib.dump(model, os.path.normpath(os.path.join('..','ml','ratefluencer_model.joblib')))
print('Saved model')